In [1]:
# -*- coding: utf-8 -*-
# CT -> MRI(b=0) Conditional Diffusion (DDPM + DDIM)
# - 데이터: /data/ct-mri/dataset/processed_final_5/{bleed,non_bleed}/CNUH_***/...
# - 모델: UNetModel (concat conditioning: x_t(1) + CT(1) -> in_channels=2)
# - 학습: FP32(AMP OFF) + EMA
# - 에폭마다: CT-only inference + 디버그(x_t -> x0hat) 저장


import os, glob, time, random, warnings, math
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


from ldm.modules.diffusionmodules.openaimodel import UNetModel
from ldm.modules.diffusionmodules.util import make_beta_schedule, make_ddim_timesteps, make_ddim_sampling_parameters

In [33]:

# ----------------------------
# Config
# ----------------------------
CFG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # 데이터 루트
    "data_root": "/data/ct-mri/dataset/processed_final_5",
    "use_groups": ["bleed", "non_bleed"],

    # 환자 폴더 내 후보 폴더/파일 패턴(소문자 비교; 파일 패턴은 glob 그대로)
    "ct_dir_candidates": ["ct_axial", "ct", "ct_png", "ctjpg", "ctjpeg"],
    "mri_b0_dir_candidates": ["mri_b0_axial", "mri_b0", "b0", "dwi_b0", "dwi", "dwi_axial_b0"],
    "ct_file_globs": ["*ct*.jpg", "*CT*.jpg"],
    "mri_b0_file_globs": ["*b0*.jpg", "*B0*.jpg", "*dwi*.jpg", "*DWI*.jpg"],

    # 학습 설정
    "img_size": 256,
    "batch_size": 8,
    "num_workers": 4,
    "epochs": 100,
    "lr": 2e-4,
    "ema_decay": 0.999,

    # Diffusion 스케줄
    "num_timesteps": 1000,
    "beta_schedule": "cosine",   # "linear" / "cosine"
    "ddim_steps": 50,
    "ddim_eta": 0.0,             # 0=결정론

    # UNet 설정
    "unet": {
        "in_channels": 2,           # x_t(1) + CT(1)
        "out_channels": 1,          # epsilon 예측
        "model_channels": 128,
        "num_res_blocks": 2,
        "channel_mult": (1,2,4,4),
        "attention_resolutions": (16,),   # 256 기준 16x
        "num_heads": -1,                  # <- 둘 중 하나 지정 필요
        "num_head_channels": 64,          # ✅ 헤드 채널 지정
        "num_heads_upsample": -1,
        "dropout": 0.1,
        "use_checkpoint": False,
        "use_fp16": False,
        "use_scale_shift_norm": True,
        "resblock_updown": False,
        "use_new_attention_order": False,
        "use_spatial_transformer": False,
        "transformer_depth": 1,
        "context_dim": None,
        "n_embed": None,
        "legacy": True,
    },

    # 체크포인트
    "save_dir": "./pixel_diff_ckpt_ct2mri_b0",
    "resume": None,   # 예: "./ckpt_ct2mri_b0/last.pt" (없으면 새로 시작)

    # 에폭별 CT-only inference
    "inference": {
        "enabled": True,
        "num_examples": 8,
        "sampler": "ddim",  # "ddim" or "ddpm"
        "ddim_steps": 100,
        "eta": 0.0,         # DDIM eta
        "seed": 1234,       # 재현성
        "ct_list": [],      # 지정 경로 리스트 (비워두면 dataset 앞에서 N장)
        "ct_glob": None,    # 예 "/data/val_ct/*.jpg"
        "out_subdir": "epoch_infer",
        "write_mapping": True  # CT stem -> 원본 경로 매핑 저장
    }
}


In [34]:
# ----------------------------
# Utils
# ----------------------------
def set_seed(s: int):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def imread_gray(path: str, size: int) -> np.ndarray:
    img = Image.open(path).convert("L").resize((size, size), Image.BICUBIC)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr

def to_tensor(img01: np.ndarray) -> torch.Tensor:
    x = torch.from_numpy(img01)[None, ...]  # (1,H,W)
    x = x * 2.0 - 1.0
    return x

def to_image(x: torch.Tensor) -> Image.Image:
    x = x.detach().clamp(-1,1)
    x = (x + 1.0) * 0.5
    x = (x * 255.0).round().cpu().numpy().astype(np.uint8)[0]
    return Image.fromarray(x, mode="L")

def save_triptych(ct_t: torch.Tensor, pred_t: torch.Tensor, gt_path: Optional[str],
                  out_path: str, size: int):
    """
    ct_t, pred_t: (1,H,W) 텐서 [-1,1]
    gt_path: GT 경로(없으면 None)
    out_path: 저장 경로(.png)
    """
    def tensor_to_pil(x):
        x = x.detach().clamp(-1,1)
        x = (x + 1.0) * 0.5
        x = (x * 255.0).round().cpu().numpy().astype(np.uint8)[0]
        return Image.fromarray(x, mode="L")

    ct_pil  = tensor_to_pil(ct_t)
    pred_pil= tensor_to_pil(pred_t)

    if gt_path and os.path.exists(gt_path):
        gt_arr = imread_gray(gt_path, size)
        gt_pil = Image.fromarray((gt_arr*255).astype(np.uint8), mode="L")
    else:
        gt_pil = Image.new("L", (size, size), color=0)

    panel = Image.new("L", (size*3, size))
    panel.paste(ct_pil,   (0,       0))
    panel.paste(gt_pil,   (size,    0))
    panel.paste(pred_pil, (size*2,  0))
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    panel.save(out_path)

def _find_first_existing_dir(base: str, candidates_lower: List[str]) -> str:
    entries = [d for d in glob.glob(os.path.join(base, "*")) if os.path.isdir(d)]
    name2path = {os.path.basename(p).lower(): p for p in entries}
    for cand in candidates_lower:
        if cand in name2path:
            return name2path[cand]
    return ""

def _glob_files(base: str, patterns: List[str]) -> List[str]:
    out = []
    for pat in patterns:
        out.extend(glob.glob(os.path.join(base, pat)))
    out = sorted(list(set(out)))
    return out

def _collect_modality_files(patient_dir: str,
                            dir_candidates: List[str],
                            file_globs: List[str]) -> List[str]:
    found_dir = _find_first_existing_dir(patient_dir, [c.lower() for c in dir_candidates])
    if found_dir:
        files = sorted(glob.glob(os.path.join(found_dir, "*.jpg")))
        if len(files) > 0:
            return files
    files = _glob_files(patient_dir, file_globs)
    return files

def _pair_by_filename(ct_files: List[str], mri_files: List[str]) -> List[Tuple[str,str]]:
    mri_map = {os.path.splitext(os.path.basename(p))[0]: p for p in mri_files}
    pairs = []
    for c in ct_files:
        stem = os.path.splitext(os.path.basename(c))[0]
        if stem in mri_map:
            pairs.append((c, mri_map[stem]))
    return pairs

def _pair_by_index(ct_files: List[str], mri_files: List[str]) -> List[Tuple[str,str]]:
    n = min(len(ct_files), len(mri_files))
    if n == 0:
        return []
    warnings.warn(f"[Pairing by index] filename match failed. Use sorted index pairing: {n} pairs.")
    return list(zip(sorted(ct_files)[:n], sorted(mri_files)[:n]))

def discover_pairs(data_root: str,
                   use_groups: List[str],
                   ct_dir_candidates: List[str],
                   mri_b0_dir_candidates: List[str],
                   ct_file_globs: List[str],
                   mri_b0_file_globs: List[str]) -> List[Tuple[str,str]]:
    all_pairs = []
    for g in use_groups:
        group_dir = os.path.join(data_root, g)
        if not os.path.isdir(group_dir):
            warnings.warn(f"[Skip] group not found: {group_dir}")
            continue
        patients = sorted([p for p in glob.glob(os.path.join(group_dir, "*")) if os.path.isdir(p)])
        for pd in patients:
            ct_files = _collect_modality_files(pd, ct_dir_candidates, ct_file_globs)
            mri_files = _collect_modality_files(pd, mri_b0_dir_candidates, mri_b0_file_globs)
            if len(ct_files) == 0 or len(mri_files) == 0:
                warnings.warn(f"[Skip patient] CT={len(ct_files)}, MRI(b0)={len(mri_files)} | {pd}")
                continue
            pairs = _pair_by_filename(ct_files, mri_files)
            if len(pairs) == 0:
                pairs = _pair_by_index(ct_files, mri_files)
            all_pairs.extend(pairs)
    if len(all_pairs) == 0:
        raise RuntimeError(f"No CT-MRI(b0) pairs discovered under {data_root}")
    return all_pairs

In [35]:
# ----------------------------
# Dataset
# ----------------------------
class CTMRIFolderDataset(Dataset):
    def __init__(self, cfg: Dict):
        self.cfg = cfg
        self.pairs = discover_pairs(
            cfg["data_root"], cfg["use_groups"],
            cfg["ct_dir_candidates"], cfg["mri_b0_dir_candidates"],
            cfg["ct_file_globs"], cfg["mri_b0_file_globs"]
        )
        self.size = cfg["img_size"]
        print(f"[Dataset] discovered pairs: {len(self.pairs)}")

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        ct_path, mri_path = self.pairs[idx]
        ct = to_tensor(imread_gray(ct_path, self.size))    # (1,H,W) [-1,1]
        mri = to_tensor(imread_gray(mri_path, self.size))  # (1,H,W) [-1,1]
        return {"ct": ct, "mri": mri, "ct_path": ct_path, "mri_path": mri_path}

In [36]:
# ----------------------------
# Diffusion schedule
# ----------------------------
@dataclass
class DiffusionSchedule:
    betas: torch.Tensor
    alphas: torch.Tensor
    alphas_cumprod: torch.Tensor
    alphas_cumprod_prev: torch.Tensor
    sqrt_alphas_cumprod: torch.Tensor
    sqrt_one_minus_alphas_cumprod: torch.Tensor
    posterior_variance: torch.Tensor

def build_schedule(num_steps: int, beta_schedule="cosine", device="cpu") -> DiffusionSchedule:
    betas = torch.from_numpy(make_beta_schedule(beta_schedule, num_steps)).float().to(device)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)

    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus = torch.sqrt(1.0 - alphas_cumprod)
    posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
    return DiffusionSchedule(
        betas, alphas, alphas_cumprod, alphas_cumprod_prev,
        sqrt_alphas_cumprod, sqrt_one_minus, posterior_variance
    )

In [37]:
# ----------------------------
# Model (concat-conditioning)
# ----------------------------
def build_model(cfg):
    u = cfg["unet"]
    net = UNetModel(
        image_size=cfg["img_size"],
        in_channels=u["in_channels"],
        out_channels=u["out_channels"],
        model_channels=u["model_channels"],
        num_res_blocks=u["num_res_blocks"],
        channel_mult=u["channel_mult"],
        attention_resolutions=u["attention_resolutions"],
        num_heads=u["num_heads"],
        num_head_channels=u["num_head_channels"],
        num_heads_upsample=u["num_heads_upsample"],
        dropout=u["dropout"],
        conv_resample=True,
        dims=2,
        num_classes=None,
        use_checkpoint=u["use_checkpoint"],
        use_fp16=u["use_fp16"],
        use_scale_shift_norm=u["use_scale_shift_norm"],
        resblock_updown=u["resblock_updown"],
        use_new_attention_order=u["use_new_attention_order"],
        use_spatial_transformer=u["use_spatial_transformer"],
        transformer_depth=u["transformer_depth"],
        context_dim=u["context_dim"],
        n_embed=u["n_embed"],
        legacy=u["legacy"]
    )
    return net

In [38]:
# ----------------------------
# EMA
# ----------------------------
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.data.clone()

    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = (1.0 - self.decay) * p.data + self.decay * self.shadow[n]

    def apply_shadow(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data = self.shadow[n]

    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data = self.backup[n]
        self.backup = {}

In [39]:

# ----------------------------
# Diffusion ops
# ----------------------------
def q_sample(x0, t, sched: DiffusionSchedule, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ab = sched.sqrt_alphas_cumprod[t].view(-1,1,1,1)
    sqrt_one_minus = sched.sqrt_one_minus_alphas_cumprod[t].view(-1,1,1,1)
    return sqrt_ab * x0 + sqrt_one_minus * noise, noise

@torch.no_grad()
def sample_ddpm(model, ct, H, sched: DiffusionSchedule, device="cpu"):
    B = ct.size(0)
    x = torch.randn(B, 1, H, H, device=device)
    T = sched.betas.size(0)
    for t in reversed(range(T)):
        t_b = torch.full((B,), t, device=device, dtype=torch.long)
        x_in = torch.cat([x, ct], dim=1).float()
        eps = model(x_in, t_b)

        alpha_bar_t = sched.alphas_cumprod[t]
        x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * eps) / torch.sqrt(alpha_bar_t)
        mean = torch.sqrt(sched.alphas_cumprod_prev[t]) * x0_pred + \
               torch.sqrt(1.0 - sched.alphas_cumprod_prev[t]) * eps

        if t > 0:
            x = mean + torch.sqrt(sched.posterior_variance[t]) * torch.randn_like(x)
        else:
            x = mean
    return x

@torch.no_grad()
def sample_ddim(model, ct, H, sched: DiffusionSchedule, ddim_steps=50, eta=0.0, device="cpu"):
    ddim_ts = make_ddim_timesteps('uniform', ddim_steps, sched.betas.size(0))
    ddim_ts = np.array(ddim_ts) - 1
    sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(
        sched.alphas_cumprod.detach().cpu().numpy(), ddim_ts, eta
    )
    sigmas = torch.from_numpy(sigmas).float().to(device)
    alphas = torch.from_numpy(alphas).float().to(device)
    alphas_prev = torch.from_numpy(alphas_prev).float().to(device)

    B = ct.size(0)
    x = torch.randn(B, 1, H, H, device=device)
    for i in reversed(range(len(ddim_ts))):
        t = int(ddim_ts[i])
        t_b = torch.full((B,), t, device=device, dtype=torch.long)
        x_in = torch.cat([x, ct], dim=1).float()
        eps = model(x_in, t_b)

        a_t = alphas[i]
        a_prev = alphas_prev[i]
        sigma_t = sigmas[i]

        x0_pred = (x - torch.sqrt(1.0 - a_t) * eps) / torch.sqrt(a_t)
        dir_term = torch.sqrt(1.0 - a_prev - sigma_t**2) * eps
        noise = sigma_t * torch.randn_like(x) if i > 0 else 0.0
        x = torch.sqrt(a_prev) * x0_pred + dir_term + noise
    return x

In [40]:
# ----------------------------
# Debug: x_t -> x0_hat 재구성
# ----------------------------
@torch.no_grad()
def debug_one_step_recon(net, sched, batch, device, save_dir, tag="dbg"):
    net.eval()
    ct = batch["ct"].to(device)[:4]
    mri = batch["mri"].to(device)[:4]
    B, _, H, W = mri.shape

    t = torch.full((B,), sched.betas.size(0)//2, device=device, dtype=torch.long)  # 중간 시점
    noise = torch.randn_like(mri)
    x_t = sched.sqrt_alphas_cumprod[t].view(-1,1,1,1) * mri + \
          sched.sqrt_one_minus_alphas_cumprod[t].view(-1,1,1,1) * noise

    x_in = torch.cat([x_t, ct], dim=1).float()
    eps = net(x_in, t)

    a_bar = sched.alphas_cumprod[t].view(-1,1,1,1)
    x0_hat = (x_t - torch.sqrt(1 - a_bar) * eps) / torch.sqrt(a_bar)

    os.makedirs(save_dir, exist_ok=True)

    def to_img(x):
        x = x.clamp(-1,1); x = (x+1)*0.5
        return (x*255).round().detach().cpu().numpy().astype("uint8")

    for i in range(B):
        Image.fromarray(to_img(mri[i,0])).save(os.path.join(save_dir, f"{tag}_gt_{i}.png"))
        Image.fromarray(to_img(ct[i,0])).save (os.path.join(save_dir, f"{tag}_ct_{i}.png"))
        Image.fromarray(to_img(x_t[i,0])).save (os.path.join(save_dir, f"{tag}_xt_{i}.png"))
        Image.fromarray(to_img(x0_hat[i,0])).save(os.path.join(save_dir, f"{tag}_x0hat_{i}.png"))


In [41]:
# ----------------------------
# Epoch inference helpers (Triptych 포함)
# ----------------------------
def _prepare_infer_cts(cfg, ds, device):
    inf = cfg["inference"]; imgs, stems = [], []
    if inf["ct_list"]:
        paths = inf["ct_list"]
    elif inf["ct_glob"]:
        paths = sorted(glob.glob(inf["ct_glob"]))[:inf["num_examples"]]
    else:
        n = min(inf["num_examples"], len(ds))
        paths = [ds.pairs[i][0] for i in range(n)]
    for p in paths:
        arr = imread_gray(p, cfg["img_size"]); imgs.append(to_tensor(arr))
        stems.append(os.path.splitext(os.path.basename(p))[0])
    ct = torch.stack(imgs, dim=0).to(device)
    return ct, stems, paths  # 경로도 반환

@torch.no_grad()
def run_epoch_inference(net, sched, cfg, ds, device, epoch_idx):
    inf = cfg["inference"]
    if not inf.get("enabled", False):
        return
    if inf["seed"] is not None:
        torch.manual_seed(inf["seed"]); torch.cuda.manual_seed_all(inf["seed"])

    ct, stems, paths = _prepare_infer_cts(cfg, ds, device)
    H = cfg["img_size"]
    outdir = os.path.join(cfg["save_dir"], inf["out_subdir"])
    os.makedirs(outdir, exist_ok=True)

    # CT->GT 매핑(dict)
    ct2gt = dict(ds.pairs)  # {ct_path: mri_path}

    # 생성
    if inf["sampler"].lower() == "ddpm":
        pred = sample_ddpm(net, ct, H, sched, device=device)
        sampler_tag = "ddpm"
    else:
        pred = sample_ddim(net, ct, H, sched, ddim_steps=inf["ddim_steps"], eta=inf["eta"], device=device)
        sampler_tag = f"ddim_eta{inf['eta']:.1f}"

    # 매핑 저장(옵션)
    if inf.get("write_mapping", False):
        with open(os.path.join(outdir, f"ep{epoch_idx:03d}_mapping.txt"), "w") as f:
            for s, p in zip(stems, paths):
                f.write(f"{s} -> {p}\n")

    # 개별 Pred + Triptych 저장
    for i, s in enumerate(stems):
        pred_path = os.path.join(outdir, f"ep{epoch_idx:03d}_{s}_{sampler_tag}.png")
        to_image(pred[i]).save(pred_path)

        gt_path = ct2gt.get(paths[i], None)
        trip_path = os.path.join(outdir, f"ep{epoch_idx:03d}_{s}_{sampler_tag}_triptych.png")
        save_triptych(ct[i], pred[i], gt_path, trip_path, size=H)

In [42]:

# ----------------------------
# Save/Load
# ----------------------------
def save_checkpoint(model, ema, opt, epoch, cfg, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "model": model.state_dict(),
        "ema": None if ema is None else ema.shadow,
        "opt": opt.state_dict(),
        "epoch": epoch,
        "cfg": cfg,
    }, path)

def load_checkpoint(model, ema, opt, path, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    if ema is not None and ckpt.get("ema", None) is not None:
        for n, p in model.named_parameters():
            if p.requires_grad and n in ckpt["ema"]:
                ema.shadow[n] = ckpt["ema"][n].clone()
    if opt is not None and "opt" in ckpt:
        opt.load_state_dict(ckpt["opt"])
    return ckpt.get("epoch", 0)

In [43]:
# ----------------------------
# Main
# ----------------------------
def main(cfg=CFG):
    set_seed(cfg["seed"])
    device = cfg["device"]
    os.makedirs(cfg["save_dir"], exist_ok=True)

    # Auto-resume 옵션(원하면 주석 해제)
    auto_resume = os.path.join(cfg["save_dir"], "last.pt")
    if cfg.get("resume") is None and os.path.exists(auto_resume):
        cfg["resume"] = auto_resume

    # data
    ds = CTMRIFolderDataset(cfg)
    dl = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True,
                    num_workers=cfg["num_workers"], pin_memory=True, drop_last=True)

    # model/sched/opt
    net = build_model(cfg).to(device)
    sched = build_schedule(cfg["num_timesteps"], cfg["beta_schedule"], device)
    opt = optim.AdamW(net.parameters(), lr=cfg["lr"])
    ema = EMA(net, decay=cfg["ema_decay"])

    start_epoch = 0
    if cfg["resume"]:
        start_epoch = load_checkpoint(net, ema, opt, cfg["resume"], device)
        print(f"[Resume] {cfg['resume']} @ epoch {start_epoch}")

    mse = nn.MSELoss()

    # ----- FP32 훈련 루프 (AMP OFF) -----
    net.train()
    for epoch in range(start_epoch, cfg["epochs"]):
        t0 = time.time()
        for step, batch in enumerate(dl):
            ct = batch["ct"].to(device)      # (B,1,H,W)
            mri = batch["mri"].to(device)    # (B,1,H,W)

            t = torch.randint(0, cfg["num_timesteps"], (ct.size(0),), device=device).long()
            x_t, noise = q_sample(mri, t, sched)

            x_in = torch.cat([x_t, ct], dim=1).float()   # FP32 강제
            eps_pred = net(x_in, t)
            loss = mse(eps_pred, noise)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            ema.update(net)

            if (step + 1) % 50 == 0:
                print(f"[{epoch+1}/{cfg['epochs']}] step {step+1}/{len(dl)}  loss {loss.item():.4f}")

        # save + mini sample
        save_checkpoint(net, ema, opt, epoch+1, cfg, os.path.join(cfg["save_dir"], "last.pt"))

        ema.apply_shadow(net); net.eval()
        with torch.no_grad():
            n_show = min(4, len(ds))
            sample_ct = torch.stack([ds[i]["ct"] for i in range(n_show)], dim=0).to(device)
            out_ddpm = sample_ddpm(net, sample_ct, cfg["img_size"], sched, device=device)
            out_ddim = sample_ddim(net, sample_ct, cfg["img_size"], sched,
                                   ddim_steps=cfg["ddim_steps"], eta=cfg["ddim_eta"], device=device)
            for i in range(n_show):
                to_image(out_ddpm[i]).save(os.path.join(cfg["save_dir"], f"ep{epoch+1:03d}_ddpm_{i}.png"))
                to_image(out_ddim[i]).save(os.path.join(cfg["save_dir"], f"ep{epoch+1:03d}_ddim_{i}.png"))

        # ✅ 에폭별 CT-only inference (EMA 가중치, Triptych 포함)
        run_epoch_inference(net, sched, cfg, ds, device, epoch+1)

        # 디버그: x_t -> x0hat (EMA 가중치 상태에서)
        batch_dbg = next(iter(dl))
        debug_one_step_recon(net, sched, batch_dbg, device, cfg["save_dir"], tag=f"ep{epoch+1:03d}")

        # EMA 복구 & 다음 에폭
        net.train(); ema.restore(net)
        print(f"Epoch {epoch+1} done in {time.time()-t0:.1f}s")

    print("Training finished.")

In [44]:
# CFG["resume"] = "./ckpt_ct2mri_b0/last.pt" 
# main(CFG)

main(CFG)

[Dataset] discovered pairs: 9533


/tmp/ipykernel_2641200/3352826770.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=device)


[Resume] ./ckpt_ct2mri_b0/last.pt @ epoch 50
[51/100] step 50/1191  loss 0.0124
[51/100] step 100/1191  loss 0.0155
[51/100] step 150/1191  loss 0.0175
[51/100] step 200/1191  loss 0.0372
[51/100] step 250/1191  loss 0.0091
[51/100] step 300/1191  loss 0.0206
[51/100] step 350/1191  loss 0.0251
[51/100] step 400/1191  loss 0.0249
[51/100] step 450/1191  loss 0.0163
[51/100] step 500/1191  loss 0.0110
[51/100] step 550/1191  loss 0.0062
[51/100] step 600/1191  loss 0.0154
[51/100] step 650/1191  loss 0.0068
[51/100] step 700/1191  loss 0.0435
[51/100] step 750/1191  loss 0.0318
[51/100] step 800/1191  loss 0.0079
[51/100] step 850/1191  loss 0.0295
[51/100] step 900/1191  loss 0.0211
[51/100] step 950/1191  loss 0.0360
[51/100] step 1000/1191  loss 0.0309
[51/100] step 1050/1191  loss 0.0089
[51/100] step 1100/1191  loss 0.0247
[51/100] step 1150/1191  loss 0.0250
Selected timesteps for ddim sampler: [  1  21  41  61  81 101 121 141 161 181 201 221 241 261 281 301 321 341
 361 381 401 4

KeyboardInterrupt: 

## Inference

In [53]:
# ==== Inference-only (EVAL root) ============================================
# eval 폴더: /data/ct-mri/dataset/processed_final_5/eval/<ID>/{ct_axial,mri_b0_axial,mri_b1000_axial}
# - ckpt에서 cfg 로드 → 모델/스케줄 재구성
# - eval/*/ct_axial/*.jpg 대상으로 CT-only inference
# - Pred + Triptych(Input/GT/Pred) 저장
# ===========================================================================

In [25]:
INFER = {
    "save_dir": "./pixel_diff_ckpt_ct2mri_b0",
    "ckpt_path": None,  # None -> save_dir/last.pt

    # ✅ eval 전용 루트/글롭
    "eval_root": "/data/ct-mri/dataset/processed_final_5/eval",
    "ct_glob": "/data/ct-mri/dataset/processed_final_5/eval/*/ct_axial/*.jpg",
    "ct_list": [],  # 지정하면 이 리스트 우선

    # 샘플러
    "sampler": "ddim",      # "ddim" or "ddpm"
    "ddim_steps": 100,
    "eta": 0.0,             # 0=결정론 (다양성↑ 원하면 0.1~0.3)
    "seed": 2025,

    "batch_size": 4,
    "out_subdir": "inference_eval"
}


In [26]:
import glob
print("CT matches:", len(glob.glob(INFER["ct_glob"])))

CT matches: 93


In [27]:
# ----------------------------
# Utilities
# ----------------------------
def set_seed(s: int):
    random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def imread_gray(path: str, size: int) -> np.ndarray:
    img = Image.open(path).convert("L").resize((size, size), Image.BICUBIC)
    return np.asarray(img, dtype=np.float32) / 255.0

def to_tensor(img01: np.ndarray) -> torch.Tensor:
    x = torch.from_numpy(img01)[None, ...]
    return x * 2.0 - 1.0

def to_image(x: torch.Tensor) -> Image.Image:
    x = x.detach().clamp(-1,1); x = (x+1.0)*0.5
    return Image.fromarray((x[0].cpu().numpy()*255).round().astype(np.uint8), "L")

def save_triptych(ct_t: torch.Tensor, pred_t: torch.Tensor, gt_path: Optional[str],
                  out_path: str, size: int):
    def t2p(u):
        u = (u.clamp(-1,1)+1)*0.5
        return Image.fromarray((u[0].cpu().numpy()*255).round().astype(np.uint8), "L")
    ct_p, pred_p = t2p(ct_t), t2p(pred_t)
    if gt_path and os.path.exists(gt_path):
        gt_p = Image.fromarray((imread_gray(gt_path, size)*255).astype(np.uint8), "L")
    else:
        gt_p = Image.new("L", (size, size), 0)
    panel = Image.new("L", (size*3, size))
    panel.paste(ct_p,(0,0)); panel.paste(gt_p,(size,0)); panel.paste(pred_p,(size*2,0))
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    panel.save(out_path)

In [28]:
# ----------------------------
# Build model & schedule from ckpt.cfg
# ----------------------------
def build_model_from_cfg(cfg):
    u = cfg["unet"]
    return UNetModel(
        image_size=cfg["img_size"],
        in_channels=u["in_channels"], out_channels=u["out_channels"],
        model_channels=u["model_channels"], num_res_blocks=u["num_res_blocks"],
        channel_mult=u["channel_mult"], attention_resolutions=u["attention_resolutions"],
        num_heads=u.get("num_heads",-1), num_head_channels=u.get("num_head_channels",64),
        num_heads_upsample=u.get("num_heads_upsample",-1), dropout=u["dropout"],
        conv_resample=True, dims=2, num_classes=None, use_checkpoint=u["use_checkpoint"],
        use_fp16=u["use_fp16"], use_scale_shift_norm=u.get("use_scale_shift_norm",True),
        resblock_updown=u.get("resblock_updown",False), use_new_attention_order=u.get("use_new_attention_order",False),
        use_spatial_transformer=u.get("use_spatial_transformer",False), transformer_depth=u.get("transformer_depth",1),
        context_dim=u.get("context_dim",None), n_embed=u.get("n_embed",None), legacy=u.get("legacy",True)
    )

@torch.no_grad()
def build_schedule_from_cfg(cfg, device):
    betas = torch.from_numpy(make_beta_schedule(cfg["beta_schedule"], cfg["num_timesteps"])).float().to(device)
    alphas = 1.0 - betas
    a_bar = torch.cumprod(alphas, dim=0)
    a_bar_prev = torch.cat([torch.tensor([1.0], device=device), a_bar[:-1]], dim=0)
    return {
        "betas": betas, "alphas": alphas,
        "alphas_cumprod": a_bar, "alphas_cumprod_prev": a_bar_prev,
        "sqrt_alphas_cumprod": torch.sqrt(a_bar),
        "sqrt_one_minus_alphas_cumprod": torch.sqrt(1.0 - a_bar),
        "posterior_variance": betas * (1.0 - a_bar_prev) / (1.0 - a_bar)
    }

In [29]:
# ----------------------------
# Samplers
# ----------------------------
@torch.no_grad()
def sample_ddpm(model, ct, H, sched, device="cpu"):
    B = ct.size(0)
    x = torch.randn(B,1,H,H, device=device)
    T = sched["betas"].size(0)
    for t in reversed(range(T)):
        t_b = torch.full((B,), t, device=device, dtype=torch.long)
        eps = model(torch.cat([x,ct],1).float(), t_b)
        a_bar = sched["alphas_cumprod"][t]
        x0 = (x - torch.sqrt(1 - a_bar) * eps) / torch.sqrt(a_bar)
        mean = torch.sqrt(sched["alphas_cumprod_prev"][t]) * x0 + torch.sqrt(1.0 - sched["alphas_cumprod_prev"][t]) * eps
        x = mean + (torch.sqrt(sched["posterior_variance"][t]) * torch.randn_like(x) if t>0 else 0)
    return x

@torch.no_grad()
def sample_ddim(model, ct, H, sched, ddim_steps=100, eta=0.0, device="cpu"):
    ts = np.array(make_ddim_timesteps('uniform', ddim_steps, sched["betas"].size(0))) - 1
    sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(sched["alphas_cumprod"].cpu().numpy(), ts, eta)
    sigmas = torch.from_numpy(sigmas).float().to(device)
    alphas = torch.from_numpy(alphas).float().to(device)
    alphas_prev = torch.from_numpy(alphas_prev).float().to(device)
    B = ct.size(0)
    x = torch.randn(B,1,H,H, device=device)
    for i in reversed(range(len(ts))):
        t = int(ts[i])
        t_b = torch.full((B,), t, device=device, dtype=torch.long)
        eps = model(torch.cat([x,ct],1).float(), t_b)
        a_t, a_prev, s_t = alphas[i], alphas_prev[i], sigmas[i]
        x0 = (x - torch.sqrt(1.0 - a_t) * eps) / torch.sqrt(a_t)
        dir_term = torch.sqrt(1.0 - a_prev - s_t**2) * eps
        noise = s_t * torch.randn_like(x) if i>0 else 0.0
        x = torch.sqrt(a_prev) * x0 + dir_term + noise
    return x

In [30]:
# ----------------------------
# Build CT->GT(b0) map from eval root
# ----------------------------
def build_ct2gt_map_eval(eval_root: str) -> dict:
    """eval/<ID>/{ct_axial,mri_b0_axial}에서 stem 매칭으로 CT->b0 GT 매핑"""
    ct2gt = {}
    if not eval_root or not os.path.isdir(eval_root): return ct2gt
    patients = sorted([p for p in glob.glob(os.path.join(eval_root, "*")) if os.path.isdir(p)])
    for pd in patients:
        ct_dir = os.path.join(pd, "ct_axial"); b0_dir = os.path.join(pd, "mri_b0_axial")
        if not (os.path.isdir(ct_dir) and os.path.isdir(b0_dir)): continue
        cts = sorted(glob.glob(os.path.join(ct_dir, "*.jpg")))
        b0s = sorted(glob.glob(os.path.join(b0_dir, "*.jpg")))
        if not cts or not b0s: continue
        m = {os.path.splitext(os.path.basename(p))[0]: p for p in b0s}
        for c in cts:
            stem = os.path.splitext(os.path.basename(c))[0]
            if stem in m: ct2gt[c] = m[stem]
    return ct2gt

In [31]:
# ----------------------------
# Runner
# ----------------------------
def run_eval_inference(INFER):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    set_seed(INFER["seed"])

    ckpt_path = INFER["ckpt_path"] or os.path.join(INFER["save_dir"], "last.pt")
    assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"
    ckpt = torch.load(ckpt_path, map_location=device)  # cfg 포함
    cfg = ckpt["cfg"]; H = cfg["img_size"]

    # model & schedule
    net = build_model_from_cfg(cfg).to(device)
    net.load_state_dict(ckpt["model"], strict=True)
    net.eval()
    sched = build_schedule_from_cfg(cfg, device)

    # CT inputs
    if INFER["ct_list"]:
        ct_paths = INFER["ct_list"]
    else:
        ct_paths = sorted(glob.glob(INFER["ct_glob"]))
    print(f"[INFO] matched CT files: {len(ct_paths)}  pattern: {INFER['ct_glob'] if not INFER['ct_list'] else 'ct_list'}")
    if len(ct_paths)==0:
        raise RuntimeError("No CT files matched.")

    # GT map
    ct2gt = build_ct2gt_map_eval(INFER["eval_root"])

    outdir = os.path.join(INFER["save_dir"], INFER["out_subdir"])
    os.makedirs(outdir, exist_ok=True)
    mapfile = open(os.path.join(outdir, "mapping.txt"), "a")

    B = INFER.get("batch_size", 4)
    for i in range(0, len(ct_paths), B):
        batch_paths = ct_paths[i:i+B]
        imgs  = [to_tensor(imread_gray(p, H)) for p in batch_paths]
        stems = [os.path.splitext(os.path.basename(p))[0] for p in batch_paths]
        # 환자ID 추출: eval/<ID>/ct_axial/<stem>.jpg
        pids  = [os.path.basename(os.path.dirname(os.path.dirname(p))) for p in batch_paths]
        ct = torch.stack(imgs, 0).to(device)

        # sampling
        if INFER["sampler"].lower() == "ddpm":
            pred = sample_ddpm(net, ct, H, sched, device=device); tag="ddpm"
        else:
            pred = sample_ddim(net, ct, H, sched, ddim_steps=INFER["ddim_steps"], eta=INFER["eta"], device=device)
            tag = f"ddim_eta{INFER['eta']:.1f}"

        # save
        for k, (pid, stem) in enumerate(zip(pids, stems)):
            base = f"{pid}_{stem}_{tag}"
            pred_path = os.path.join(outdir, f"{base}.png")
            to_image(pred[k]).save(pred_path)

            gt_path = ct2gt.get(batch_paths[k], None)
            trip_path = os.path.join(outdir, f"{base}_triptych.png")
            save_triptych(ct[k], pred[k], gt_path, trip_path, size=H)

            mapfile.write(f"{base} <- {batch_paths[k]}\n")

    mapfile.close()
    print(f"[Done] Saved to: {outdir}")

In [32]:
run_eval_inference(INFER)

/tmp/ipykernel_2641200/1812327521.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)  # cfg 포함


[INFO] matched CT files: 93  pattern: /data/ct-mri/dataset/processed_final_5/eval/*/ct_axial/*.jpg
Selected timesteps for ddim sampler: [  1  11  21  31  41  51  61  71  81  91 101 111 121 131 141 151 161 171
 181 191 201 211 221 231 241 251 261 271 281 291 301 311 321 331 341 351
 361 371 381 391 401 411 421 431 441 451 461 471 481 491 501 511 521 531
 541 551 561 571 581 591 601 611 621 631 641 651 661 671 681 691 701 711
 721 731 741 751 761 771 781 791 801 811 821 831 841 851 861 871 881 891
 901 911 921 931 941 951 961 971 981 991]
Selected alphas for ddim sampler: a_t: [9.99958694e-01 9.99278784e-01 9.98114109e-01 9.96465683e-01
 9.94335055e-01 9.91724551e-01 9.88636315e-01 9.85073686e-01
 9.81040001e-01 9.76539135e-01 9.71575499e-01 9.66153920e-01
 9.60279524e-01 9.53958333e-01 9.47196424e-01 9.39999759e-01
 9.32376087e-01 9.24332678e-01 9.15877044e-01 9.07017410e-01
 8.97763014e-01 8.88121843e-01 8.78103852e-01 8.67719054e-01
 8.56976807e-01 8.45888078e-01 8.34463477e-01 8.2271